<a href="https://colab.research.google.com/github/idorotner222/CloudComputing/blob/main/tirgul3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/אורט בראודה - הנדסת תוכנה/cloud computing')


In [ ]:
# @title Writing to the DB

!pip install firebase


In [ ]:
# @title כתיבת נתונים רציפה ל-Firebase (מתודת Post)
from firebase import firebase
FBconn = firebase.FirebaseApplication('https://fir-tirgul3-default-rtdb.firebaseio.com/',None)
while True:
  temperature = int (input ("what is the temperature?"))
  data_to_upload = {
      'Temp' : temperature
  }
  result = FBconn.post('/myTest1/',data_to_upload)
  #result = FBconn.post(‘./',data_to_upload) if we want to write directly in root and not inside folders

  print(result)


In [ ]:
# @title Reading from DB
from firebase import firebase

firebase = firebase.FirebaseApplication('https://fir-tirgul3-default-rtdb.firebaseio.com/', None)
result = firebase.get('/myTest1/', None)
#result= object of JSON, it called Dictionary.
print(result)

In [ ]:
# @title Reading from DB and print key before
import json
from firebase import firebase
FBconn = firebase.FirebaseApplication('https://fir-tirgul3-default-rtdb.firebaseio.com/',None)
res=FBconn.get('/myTest1/',None)

for key in res:
    print(key+":\t", res[key])


In [ ]:
# @title Action on DB(create,read,update,delete,disply)
from firebase import firebase
import time

# Initialize Firebase connection
FBconn = firebase.FirebaseApplication('https://fir-tirgul3-default-rtdb.firebaseio.com/', None)

#record_id=key of JSON
def get_all_records():
    """Retrieve all temperature records"""
    return FBconn.get('/myTest1/', None)

def update_temperature(record_id, new_temp):
    """Update an existing temperature record"""
    return FBconn.put('/myTest1/' + record_id, 'Temp', new_temp)

def delete_record(record_id):
    """Delete a temperature record"""
    return FBconn.delete('/myTest1/', record_id)

def display_records():
    """Display all temperature records"""
    records = get_all_records()
    if records:
        print("\nCurrent Records:")
        for record_id, data in records.items():
            print(f"ID: {record_id} - Temperature: {data['Temp']}°C")
    else:
        print("\nNo records found")

while True:
    print("\nTemperature Tracker Menu:")
    print("1. Add new temperature")
    print("2. Update existing temperature")
    print("3. Delete temperature record")
    print("4. View all records")
    print("5. Exit")

    choice = input("\nEnter your choice (1-5): ")

    if choice == '1':
        # Add new temperature
        try:
            temperature = int(input("What is the temperature? "))
            data_to_upload = {
                'Temp': temperature,
                'timestamp': time.time()  # Adding timestamp for better tracking
            }
            result = FBconn.post('/myTest1/', data_to_upload)
            print(f"Temperature added successfully! Record ID: {result['name']}")
        except ValueError:
            print("Please enter a valid number for temperature")

    elif choice == '2':
        # Update existing temperature
        display_records()
        record_id = input("\nEnter the record ID to update: ")
        try:
            new_temp = int(input("Enter the new temperature: "))
            result = update_temperature(record_id, new_temp)
            if result:
                print("Temperature updated successfully!")
            else:
                print("Failed to update temperature. Check if the ID exists.")
        except ValueError:
            print("Please enter a valid number for temperature")

    elif choice == '3':
        # Delete temperature record
        display_records()
        record_id = input("\nEnter the record ID to delete: ")
        result = delete_record(record_id)
        if result is None:
            print("Record deleted successfully!")
        else:
            print("Failed to delete record. Check if the ID exists.")

    elif choice == '4':
        # View all records
        display_records()

    elif choice == '5':
        print("Exiting program...")
        break

    else:
        print("Invalid choice. Please try again.")


In [ ]:
# @title CSS in python
from IPython.display import HTML
from google.colab.output import _publish as publish
publish.css("b {color: blue}")
HTML("Now <b>bold</b> and <b>blue</b> too.")



In [ ]:
# @title HTML in python
import IPython

html_code = \
'''
<p>Now this is <i>really</i> awesome!</p>
<img src='https://share.google/CZDrk8dlu31U5Fa6S'></img>
'''


display(IPython.display.HTML(html_code))



In [ ]:
# @title Graphs in python
import altair as alt
import ipywidgets as widgets
from vega_datasets import data

source = data.stocks()

stock_picker = widgets.SelectMultiple(
    options=source.symbol.unique(),
    value=list(source.symbol.unique()),
    description='Symbols')

# The value of symbols will come from the stock_picker.
@widgets.interact(symbols=stock_picker)
def render(symbols):
  selected = source[source.symbol.isin(list(symbols))]

  return alt.Chart(selected).mark_line().encode(
      x='date',
      y='price',
      color='symbol',
      strokeDash='symbol',
  )



In [ ]:
# @title Take a photo in python
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

from IPython.display import Image
try:
  filename = take_photo()
  print('Saved to {}'.format(filename))

  # Show the image which was just taken.
  display(Image(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))


In [ ]:
# @title Java Script in python
import IPython
js_code = '''
document.querySelector("#output-area").appendChild(document.createTextNode("hello world!"));
'''
display(IPython.display.Javascript(js_code))



In [ ]:
# @title Add item to List
import IPython
from google.colab import output

display(IPython.display.HTML('''
    The items:
    <br><ol id="items"></ol>
    <button id='button'>Click to add</button>
    <script>
      document.querySelector('#button').onclick = () => {
        google.colab.kernel.invokeFunction('notebook.AddListItem', [], {});
      };
    </script>
    '''))

def add_list_item():
  # Use redirect_to_element to direct the elements which are being written.
  with output.redirect_to_element('#items'):
    # Use display to add items which will be persisted on notebook reload.
    display(IPython.display.HTML('<li> Another item</li>'))

output.register_callback('notebook.AddListItem', add_list_item)



In [ ]:
# @title כפתור HTML אינטראקטיבי בתוך סביבת Google Colab
import IPython
import uuid
from google.colab import output

class InvokeButton(object):
  def __init__(self, title, callback):
    self._title = title
    self._callback = callback

  def _repr_html_(self):
    callback_id = 'button-' + str(uuid.uuid4())
    output.register_callback(callback_id, self._callback)

    template = """<button id="{callback_id}">{title}</button>
        <script>
          document.querySelector("#{callback_id}").onclick = (e) => {{
            google.colab.kernel.invokeFunction('{callback_id}', [], {{}})
            e.preventDefault();
          }};
        </script>"""
    html = template.format(title=self._title, callback_id=callback_id)
    return html

def do_something():
  print('here')

InvokeButton('click me', do_something)



In [ ]:
# @title slider
import ipywidgets as widgets

slider = widgets.IntSlider(20, min=0, max=100)
slider



In [ ]:
# @title Final practice
from firebase import firebase
import time

# Initialize Firebase connection
FBconn = firebase.FirebaseApplication('https://fir-tirgul3-default-rtdb.firebaseio.com/', None)
# נגדיר את הנתיב במסד הנתונים שבו נשמור את המילים
DB_PATH = '/word_counts/'
#record_id=key of JSON
def get_all_words():
      return FBconn.get(DB_PATH, None)
def update_word_in_db(word, count):
    """עדכון כמות של מילה בודדת (אם המילה לא קיימת היא תיווצר)"""
    # המילה עצמה משמשת כמפתח, והספירה היא הערך
    return FBconn.put(DB_PATH, word, count)
def delete_word(word):
    # """מחיקת מילה ממסד הנתונים"""
      return FBconn.delete(DB_PATH, word)
def display_words():
    """הצגת כל המילים והכמויות שלהן בפורמט המבוקש"""
    words_dict = get_all_words()
    if words_dict:
        # יצירת רשימה של מחרוזות בפורמט "word:count"
        output = [f"{word}:{count}" for word, count in words_dict.items()]
        # הדפסה מחוברת עם פסיקים, בדיוק כמו בדוגמה בשקופית
        print(", ".join(output))
    else:
        print("\nNo words found in database.")

while True:
    print("\nTemperature Tracker Menu:")
    print("1. Add single word")
    print("2. Add text for analysis")
    print("3. Update word count")
    print("4. Delete word")
    print("5. View all words")
    print("6. Exit")

    choice = input("\nEnter your choice (1-5): ")

    if choice == '1':
        # הוספת מילה בודדת
        word = input("Enter a word: ").strip().lower() # הפיכה לאותיות קטנות למניעת כפילויות
        if ' ' in word:
            print("Error: Please enter only ONE word. To add text, use option 2.")
        else:
            # נביא את כל המילים כדי לבדוק אם המילה קיימת וכמה פעמים
            current_data = get_all_words() or {}
            current_count = current_data.get(word, 0)

            # נעדכן את המסד עם הספירה החדשה (+1)
            update_word_in_db(word, current_count + 1)
            print(f"Word '{word}' updated. New count: {current_count + 1}")

    elif choice == '2':
        # הוספת טקסט שלם לניתוח
        text = input("Enter text for analysis: ").strip().lower()
        words_list = text.split() # פירוק הטקסט למילים בודדות

        if words_list:
            current_data = get_all_words() or {}

            # ספירת המילים במילון המקומי
            for word in words_list:
                current_data[word] = current_data.get(word, 0) + 1

            # שליחת כל המילים המעודכנות חזרה ל-Firebase
            for word, count in current_data.items():
                update_word_in_db(word, count)

            print(f"Added/Updated {len(words_list)} words from text.")

    elif choice == '3':
        # עדכון מספר ההופעות של מילה בצורה ידנית
        display_words()
        word = input("\nEnter the word to update: ").strip().lower()
        try:
            new_count = int(input("Enter the new count: "))
            update_word_in_db(word, new_count)
            print(f"Count for '{word}' updated to {new_count}!")
        except ValueError:
            print("Please enter a valid number.")

    elif choice == '4':
        # מחיקת מילה
        display_words()
        word = input("\nEnter the word to delete: ").strip().lower()
        delete_word(word)
        print(f"Word '{word}' deleted (if it existed).")

    elif choice == '5':
        # צפייה בכל המילים השמורות
        print("\nCurrent Words:")
        display_words()

    elif choice == '6':
        print("Exiting program...")
        break

    else:
        print("Invalid choice. Please try again.")